In [17]:
%matplotlib inline
import matplotlib.pyplot as plt
import atomap.api as am
import hyperspy.api as hs
import numpy as np
import os


In [40]:
# Load the Image and RESET Metadata for Raw Pixel Analysis
datapath = 'calibrated_img_sq_on_zone.hspy'
if os.path.exists(datapath):
    s = hs.load(datapath)
    s.change_dtype('float32')
    
    # Save the original pixel size (in Angstroms) for visual scale bars
    global px_size_A
    px_size_A = s.axes_manager[0].scale
    if px_size_A == 1.0 or px_size_A <= 0.0:
        px_size_A = 0.215 # Fallback to standard Pt ptychography pixel size
    
    # Reset calibration to raw pixels (1 unit = 1 pixel)
    s.axes_manager[0].scale = 1.0
    s.axes_manager[0].offset = 0.0
    s.axes_manager[0].units = 'px'
    s.axes_manager[1].scale = 1.0
    s.axes_manager[1].offset = 0.0
    s.axes_manager[1].units = 'px'
    
    print(f"Loaded Raw Image: {s} (Pixel Size: {px_size_A:.4f} A)")
    s.plot()
    plt.show()
else:
    print(f"Error: {datapath} not found.")


In [ ]:
# Step 0.5: Interactive Polygon Region of Interest (ROI) Selection
import json
import io
import base64
import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML
from PIL import Image

# 1. Convert the 2D image data of 's' to a base64 encoded PNG to draw on HTML canvas
arr = s.data
arr_normalized = (arr - arr.min()) / (arr.max() - arr.min() + 1e-12) * 255.0
arr_uint8 = arr_normalized.astype(np.uint8)

# Save to buffer as PNG
img_pil = Image.fromarray(arr_uint8)
buf = io.BytesIO()
img_pil.save(buf, format='PNG')
base64_img = base64.b64encode(buf.getvalue()).decode('utf-8')

ny, nx = arr.shape
unique_id = "atom_roi"

# Coordinates communication widget (hidden input field style)
coords_widget = widgets.Text(value='', description='ROI Coords:', layout=widgets.Layout(width='400px'))
coords_widget.add_class(f"roi-coords-input-{unique_id}")

# HTML Drawing Canvas Interface
html_content = f"""
<div style="display: flex; flex-direction: column; align-items: flex-start; gap: 10px; padding: 15px; background: #2b2b2b; border: 1px solid #444; border-radius: 8px; color: #eee; font-family: sans-serif;">
    <h4 style="margin: 0; color: #fff;">Interactive Polygon ROI Selector</h4>
    <p style="margin: 0; font-size: 13px; color: #ccc;">Left-click to place vertices. Click 'Close Polygon' or double-click to finish drawing.</p>
    <div style="position: relative;">
        <canvas id="roi_canvas_{unique_id}" width="{nx}" height="{ny}" style="border: 1px solid #555; background-color: #000; cursor: crosshair; display: block;"></canvas>
    </div>
    <div style="display: flex; gap: 10px;">
        <button id="btn_clear_{unique_id}" style="padding: 8px 16px; background-color: #d9534f; color: white; border: none; border-radius: 4px; cursor: pointer; font-weight: bold;">Clear Polygon</button>
        <button id="btn_close_{unique_id}" style="padding: 8px 16px; background-color: #f0ad4e; color: white; border: none; border-radius: 4px; cursor: pointer; font-weight: bold;">Close Polygon</button>
    </div>
</div>

<script>
(function() {{
    const canvas = document.getElementById("roi_canvas_{unique_id}");
    const ctx = canvas.getContext("2d");
    const img = new Image();
    img.src = "data:image/png;base64,{base64_img}";
    
    let points = [];
    let isClosed = false;
    
    img.onload = function() {{
        draw();
}};
    
    function draw() {{
        ctx.clearRect(0, 0, canvas.width, canvas.height);
        ctx.drawImage(img, 0, 0);
        
        if (points.length === 0) return;
        
        ctx.strokeStyle = "#00ff00";
        ctx.lineWidth = 2.5;
        ctx.fillStyle = "rgba(0, 255, 0, 0.25)";
        
        ctx.beginPath();
        ctx.moveTo(points[0].x, points[0].y);
        for (let i = 1; i < points.length; i++) {{
            ctx.lineTo(points[i].x, points[i].y);
        }}
        if (isClosed) {{
            ctx.closePath();
            ctx.stroke();
            ctx.fill();
        }} else {{
            ctx.stroke();
        }}
        
        // Draw vertices
        ctx.fillStyle = "#00ff00";
        for (let p of points) {{
            ctx.beginPath();
            ctx.arc(p.x, p.y, 4, 0, 2 * Math.PI);
            ctx.fill();
        }}
    }}
    
    canvas.addEventListener("click", function(e) {{
        if (isClosed) return;
        const rect = canvas.getBoundingClientRect();
        const x = Math.round((e.clientX - rect.left) * (canvas.width / rect.width));
        const y = Math.round((e.clientY - rect.top) * (canvas.height / rect.height));
        points.push({{x, y}});
        updatePythonCoords();
        draw();
}});
    
    canvas.addEventListener("dblclick", function(e) {{
        if (points.length < 3) return;
        isClosed = true;
        updatePythonCoords();
        draw();
}});
    
    document.getElementById("btn_clear_{unique_id}").addEventListener("click", function() {{
        points = [];
        isClosed = false;
        updatePythonCoords();
        draw();
}});
    
    document.getElementById("btn_close_{unique_id}").addEventListener("click", function() {{
        if (points.length < 3) return;
        isClosed = true;
        updatePythonCoords();
        draw();
}});
    
    function updatePythonCoords() {{
        const inputEl = document.querySelector(".roi-coords-input-{unique_id} input");
        if (inputEl) {{
            const coordsStr = JSON.stringify(points.map(p => [p.x, p.y]));
            inputEl.value = coordsStr;
            inputEl.dispatchEvent(new Event("input", {{ bubbles: true }}));
        }}
    }}
}})();
</script>
"""

# Button to confirm and apply the ROI
confirm_btn = widgets.Button(
    description="Confirm and Create Mask",
    button_style="success",
    icon="check",
    layout=widgets.Layout(width='200px', margin='10px 0 0 0')
)
confirm_out = widgets.Output()

def on_confirm(b):
    global s_masked, polygon_coords
    with confirm_out:
        confirm_out.clear_output()
        if not coords_widget.value:
            print("Error: Please draw a polygon on the canvas first.")
            return
        
        try:
            polygon_coords = json.loads(coords_widget.value)
            if len(polygon_coords) < 3:
                print("Error: Polygon must have at least 3 vertices.")
                return
            
            # Create a 2D boolean mask from the polygon coordinates
            from matplotlib.path import Path
            y_grid, x_grid = np.mgrid[0:ny, 0:nx]
            grid_points = np.vstack((x_grid.flatten(), y_grid.flatten())).T
            path = Path(polygon_coords)
            mask = path.contains_points(grid_points).reshape((ny, nx))
            
            # Mask the signal 's': pixels outside the ROI are set to the mean background intensity
            s_masked = s.copy()
            s_masked.data = np.where(mask, s.data, np.mean(s.data))
            
            print(f"Success! Polygon ROI confirmed with {len(polygon_coords)} vertices.")
            print(f"Created 's_masked' signal containing only the ROI region.")
            
        except Exception as e:
            print(f"Error parsing coordinates: {e}")

confirm_btn.on_click(on_confirm)

display(HTML(html_content))
display(widgets.HBox([coords_widget, confirm_btn]))
display(confirm_out)


In [ ]:
# Step 1: Optimized Atom Position Discovery
print("Finding atom positions with refined thresholds...")

# --- TUNABLE PARAMETERS ---
separation_value = 4
threshold = 0.42

atom_positions = am.get_atom_positions(
    s_masked, 
    separation=separation_value, 
    threshold_rel=threshold, 
    pca=False, 
    subtract_background=True
)

# Robustly extract coordinates
if isinstance(atom_positions, np.ndarray):
    coords = atom_positions
elif hasattr(atom_positions, "atom_positions"):
    coords = np.array(atom_positions.atom_positions)
else:
    try:
        coords = np.array([[p.x, p.y] for p in atom_positions])
    except AttributeError:
        coords = np.array(atom_positions)

# Filter to keep only coordinates strictly inside the polygon ROI
if 'polygon_coords' in globals() and len(polygon_coords) >= 3:
    from matplotlib.path import Path
    polygon_path = Path(polygon_coords)
    inside_mask = polygon_path.contains_points(coords)
    coords = coords[inside_mask]
    print("Filtered coords to keep only atoms inside the polygon ROI.")

print(f"TOTAL ATOMS DETECTED: {len(coords)}")

# Visual Check: Markers should be centered on atoms
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(s.data, cmap="gray", origin="upper")
ax.scatter(coords[:, 0], coords[:, 1], color="red", s=10, marker="x")
ax.set_title(f"Detected Atoms: {len(coords)} (Check Alignment)")
plt.show()


In [ ]:
# Step 1.5: Manually Add Missing Atom Positions
# You can specify the exact (x, y) coordinates of any missing atoms here
manual_coords = [
    # [x_coordinate, y_coordinate]
    # Example: [120.5, 142.3],
]

if 'coords' in locals() and len(manual_coords) > 0:
    new_points = np.array(manual_coords)
    coords = np.vstack([coords, new_points])
    print(f"Added {len(manual_coords)} manual atom coordinates.")
    print(f"Total atom coordinates now: {len(coords)}")
    
    # Visual confirmation of the addition
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(s.data, cmap="gray", origin="upper")
    ax.scatter(coords[:, 0], coords[:, 1], color="red", s=10, marker="x")
    ax.scatter(new_points[:, 0], new_points[:, 1], color="yellow", s=60, marker="o", facecolors='none', edgecolors='yellow', label='Manual Additions')
    ax.set_title(f"Detected Atoms with Manual Additions: {len(coords)} total")
    ax.legend()
    plt.show()
else:
    print("No manual coordinates added. Keeping original coordinates from Step 1.")


### Understanding "Zone Axes" in Atomap

In Atomap, a **Zone Axis** is not defined by an external crystal model. Instead, it is found **empirically** by looking at the vectors between every atom and its neighbors. 

1. The algorithm clusters these vectors to find the most common directions and distances.
2. These common directions are labeled as Zone Axes (Zone Axis 0, 1, etc.).
3. **Lattice Spacing**: Each Zone Axis corresponds to a specific set of atomic planes with a characteristic average spacing (in pixels).
4. **Referenceless Strain**: We calculate strain by measuring local deviations from this average spacing. This allows us to see variations like expansion or compression relative to the rest of the grain without needing a theoretical reference.

In [20]:
# Step 2: Sublattice Creation & Robust Refinement
if len(coords) > 5:
    # Sublattice initialization using tolist() for compatibility
    sublattice = am.Sublattice(atom_position_list=coords.tolist(), image=s.data)

    print("Finding nearest neighbors (for fitting constraints)...")
    sublattice.find_nearest_neighbors(nearest_neighbors=15)

    print("Refining atomic positions using 2D Gaussian fitting...")
    try:
        sublattice.refine_atom_positions_using_2d_gaussian()
        print(f"Refinement complete. Atoms remaining: {len(sublattice.atom_list)}")
    except Exception as e:
        print(f"Refinement encountered an issue: {e}")
    
    # Visualization: Check if markers are now perfectly centered
    print("Plotting refined atomic positions...")
    sublattice.plot()
    plt.title('Refined Atomic Positions (Gaussian Fitted)')
    plt.show()
else:
    print("ERROR: Not enough atoms detected in Step 1 to proceed.")


### Step 2.5: Filter Atoms by Shape (Ellipticity & Sigma)

Blurred atoms or non-atomic lattice features often have high **ellipticity** (non-circular) or large **sigma** (too wide). 

1. **Ellipticity**: $\sigma_{max} / \sigma_{min}$. 1.0 is perfectly circular. Values $> 1.2$ usually indicate blurring or overlapping atoms.
2. **Sigma Average**: The average width of the atom Gaussian in pixels. Large values indicate out-of-focus or blurred atoms.

In [ ]:
if 'sublattice' in locals():
    # --- 1. Visualize current distributions ---
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    ax1.hist(sublattice.ellipticity, bins=30, color='skyblue', edgecolor='black')
    ax1.set_title("Distribution of Ellipticity")
    ax1.set_xlabel("Ellipticity (1.0 = Circular)")
    
    ax2.hist(sublattice.sigma_average, bins=30, color='salmon', edgecolor='black')
    ax2.set_title("Distribution of Sigma Average")
    ax2.set_xlabel("Sigma (px)")
    plt.show()
    
    # Optional: Plot a spatial map of ellipticity to identify problematic regions
    print("Generating spatial maps of atom shape...")
    try:
        sublattice.get_ellipticity_map().plot()
        plt.title("Ellipticity Map")
        plt.show()
        
        sublattice.get_sigma_average_map().plot()
        plt.title("Sigma Average Map")
        plt.show()
    except Exception as e:
        print(f"Map generation failed: {e}")
        print("Falling back to coordinate-based mapping...")
        x = sublattice.x_position
        y = sublattice.y_position
        sublattice.get_property_map(x, y, sublattice.ellipticity).plot()
        plt.show()
    
    # --- 2. Define Thresholds & Filter ---
    # Adjust these based on the histograms above
    max_ellipticity = 1.25
    max_sigma = 3.0
    
    ellipticity_list = np.array(sublattice.ellipticity)
    sigma_list = np.array(sublattice.sigma_average)
    
    mask = (ellipticity_list <= max_ellipticity) & (sigma_list <= max_sigma)
    
    num_before = len(sublattice.atom_list)
    # Apply filter by keeping only atoms that satisfy the mask
    sublattice.atom_list = [atom for i, atom in enumerate(sublattice.atom_list) if mask[i]]
    num_after = len(sublattice.atom_list)
    
    print(f"FILTERING COMPLETE:")
    print(f"- Atoms before: {num_before}")
    print(f"- Atoms after:  {num_after}")
    print(f"- Atoms removed: {num_before - num_after}")

    # --- Visual Check of Filtering Result ---
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(s.data, cmap="gray", origin="upper")
    # Get current positions for visualization
    final_points = np.array([p.get_pixel_position() for p in sublattice.atom_list])
    ax.scatter(final_points[:, 0], final_points[:, 1], color="red", s=10, marker="o", alpha=0.6)
    ax.set_title(f"Filtered Atoms: {len(final_points)} Remaining")
    plt.show()
    
    # Re-run neighbor finding to update the lattice connectivity after removing atoms
    sublattice.find_nearest_neighbors(nearest_neighbors=15)
else:
    print("ERROR: Sublattice not found. Run Step 2 first.")


In [21]:
# Step 3: Identify Periodic Directions & Comprehensive Analysis
if "sublattice" in locals():
    print("Clustering neighbor vectors to find major lattice directions...")
    try:
        # Identify periodic directions
        sublattice.construct_zone_axes(atom_plane_tolerance=1.0)
        
        # Get all identified directions (Zone Vectors)
        zone_keys = list(sublattice.atom_planes_by_zone_vector.keys())
        num_dirs = len(zone_keys)
        print(f"Detected {num_dirs} major periodic directions.")
        
        if num_dirs > 0:
            # Create a multi-row figure
            fig, axes = plt.subplots(num_dirs, 2, figsize=(15, 6 * num_dirs))
            # Handle the case where subplots returns a 1D array if num_dirs=1
            if num_dirs == 1:
                axes = np.array([axes])
            
            for i in range(num_dirs):
                key = zone_keys[i]
                planes = sublattice.atom_planes_by_zone_vector[key]
                avg_dist = abs(key[1])
                
                # --- COLUMN 1: VISUALIZE ATOMIC PLANES ---
                ax_plane = axes[i, 0]
                ax_plane.imshow(s.data, cmap="gray", origin="upper")
                
                # Corrected coordinate extraction using get_pixel_position()
                for plane in planes:
                    # atom.get_pixel_position() returns (x, y)
                    p_coords = np.array([atom.get_pixel_position() for atom in plane.atom_list])
                    ax_plane.plot(p_coords[:, 0], p_coords[:, 1], linewidth=1, alpha=0.7)
                    ax_plane.scatter(p_coords[:, 0], p_coords[:, 1], color="red", s=5, alpha=0.5)

                ax_plane.set_title(f"Direction {i}: Atomic Planes\n(Spacing: {avg_dist:.2f} px)")
                
                # --- COLUMN 2: STRAIN MAPPING ---
                ax_strain = axes[i, 1]
                dist_map = sublattice.get_monolayer_distance_map(
                    atom_plane_list=planes, 
                    upscale_map=1
                )
                
                strain_data = dist_map.data
                if strain_data.ndim == 3:
                    strain_data = strain_data[0] if strain_data.shape[0] > 1 else strain_data.squeeze()
                
                strain_map = (strain_data - avg_dist) / avg_dist * 100
                
                im = ax_strain.imshow(strain_map, cmap="RdBu_r", vmin=-5, vmax=5)
                ax_strain.set_title(f"Direction {i}: Strain Map [%]")
                plt.colorbar(im, ax=ax_strain, label="Strain [%]")
            
            plt.suptitle("Full Lattice Analysis: Planes vs. Local Strain", fontsize=16, y=1.02)
            plt.tight_layout()
            plt.show()
        else:
            print("Warning: No lattice directions identified.")
    except Exception as e:
        print(f"Lattice reconstruction failed: {e}")
else:
    print("ERROR: Sublattice not found. Run Step 2 first.")


In [ ]:
# Step 4: Radial & Tangential Strain Vector Analysis
print("Calculating radial strain and generating vector maps...")

if "sublattice" in locals():
    try:
        from scipy.spatial import Delaunay
        from scipy.interpolate import griddata
        
        # 1. Get refined atomic coordinates
        points = np.array([p.get_pixel_position() for p in sublattice.atom_list])
        if len(points) < 10: raise ValueError("Insufficient atoms for analysis.")
        
        # 2. Compute the centroid (center of mass) of the nanoparticle
        xc = np.mean(points[:, 0])
        yc = np.mean(points[:, 1])
        print(f"Nanoparticle center located at: ({xc:.2f}, {yc:.2f})")
        
        # 3. Create Delaunay mesh
        tri = Delaunay(points)
        tri_centers = points[tri.simplices].mean(axis=1)
        
        # 4. Calculate local Cartesian strain components on triangles (Cartesian proxy)
        exx_list = []
        eyy_list = []
        for simplex in tri.simplices:
            nodes = points[simplex]
            exx_list.append(np.std(nodes[:, 0]) / np.mean(np.abs(nodes[:, 0] + 1e-6)))
            eyy_list.append(np.std(nodes[:, 1]) / np.mean(np.abs(nodes[:, 1] + 1e-6)))
        
        # Normalize to get local variation around the mean
        exx_norm = (np.array(exx_list) - np.mean(exx_list))
        eyy_norm = (np.array(eyy_list) - np.mean(eyy_list))
        
        # 5. Interpolate Cartesian strain components back onto individual atom points
        exx_atoms = griddata(tri_centers, exx_norm, points, method="linear", fill_value=0)
        eyy_atoms = griddata(tri_centers, eyy_norm, points, method="linear", fill_value=0)
        
        # 6. Project Cartesian strain onto the Radial unit vector at each atom site
        err_list = []
        u_arrow = []
        v_arrow = []
        
        for i in range(len(points)):
            x, y = points[i]
            dx = x - xc
            dy = y - yc
            r = np.sqrt(dx**2 + dy**2 + 1e-12)
            
            # Unit radial vector components
            ux = dx / r
            uy = dy / r
            
            # Projected radial strain: Err = Exx * cos^2(phi) + Eyy * sin^2(phi)
            err = exx_atoms[i] * (ux**2) + eyy_atoms[i] * (uy**2)
            err_list.append(err)
            
            # Define arrow pointing radially outward (for tension, err > 0) or inward (for compression, err < 0)
            u_arrow.append(err * ux)
            v_arrow.append(err * uy)
            
        err_list = np.array(err_list)
        u_arrow = np.array(u_arrow)
        v_arrow = np.array(v_arrow)
        
        # 7. Visualization: Single Quiver Plot
        fig, ax = plt.subplots(figsize=(10, 8))
        
        # Quiver Plot with Radial Vector Arrows overlaid on raw image
        ax.imshow(s.data, cmap="gray", origin="upper")
        ax.scatter(points[:, 0], points[:, 1], color="gray", s=5, alpha=0.5, label="Atoms")
        ax.scatter([xc], [yc], color="yellow", marker="*", s=150, edgecolor="black", label="Center")
        
        # Draw arrows colored by radial strain using user configuration
        q = ax.quiver(points[:, 0], points[:, 1], u_arrow, v_arrow, err_list * 100,
                       cmap="turbo", angles="xy", scale_units="xy", scale=0.01,
                       width=0.008, headwidth=4, headlength=5, headaxislength=4.5,
                       clim=(-5, 5))
        
        plt.colorbar(q, ax=ax, fraction=0.038, pad=0.04, label="Radial Strain [%]")
        ax.legend()
        ax.grid(False)
        ax.axis('off')
        
        # --- ADD 2 NM SCALE BAR ---
        px_val = globals().get('px_size_A', 0.215)
        ny, nx = s.data.shape
        bar_w = 20.0 / px_val # 2 nm = 20 Angstroms
        bar_h = max(1.0, ny * 0.02)
        
        x_pos = nx - (nx * 0.08) - bar_w
        y_pos = ny - (ny * 0.08) - bar_h
        
        from matplotlib.patches import Rectangle
        rect = Rectangle((x_pos, y_pos), bar_w, bar_h, color='white', fill=True, zorder=5)
        ax.add_patch(rect)
        
        ax.text(x_pos + bar_w/2.0, y_pos - (ny * 0.015), "2 nm", 
                 color='white', fontsize=12, fontweight='bold', 
                 ha='center', va='bottom', zorder=5)
        
        ax.set_title("Radial Strain Vectors (Expansion = Outward, Contraction = Inward)", fontsize=12)
        plt.tight_layout()
        plt.show()

    except Exception as e:
        print(f"Radial strain analysis failed: {e}")
else:
    print("ERROR: Sublattice not found.")


In [ ]:
# Visualizing Delaunay Triangulation & Vacancy Triangles
if "sublattice" in locals():
    try:
        from scipy.spatial import Delaunay
        import matplotlib.pyplot as plt
        
        # 1. Get refined atomic coordinates
        points = np.array([p.get_pixel_position() for p in sublattice.atom_list])
        if len(points) < 3: raise ValueError("Insufficient atoms for triangulation.")
        
        # 2. Compute Delaunay Triangulation
        tri = Delaunay(points)
        
        # 3. Visualization
        fig, ax = plt.subplots(figsize=(10, 8))
        # Draw the raw ptychography image
        ax.imshow(s.data, cmap="gray", origin="upper")
        
        # Draw Delaunay mesh edges in cyan
        ax.triplot(points[:, 0], points[:, 1], tri.simplices, color="cyan", linewidth=1.2, alpha=0.8, label="Delaunay Mesh")
        
        # Draw detected atom sites as red circles
        ax.scatter(points[:, 0], points[:, 1], color="red", s=30, zorder=3, label="Detected Atoms")
        
        # Mark the nanoparticle centroid
        xc = np.mean(points[:, 0])
        yc = np.mean(points[:, 1])
        ax.scatter([xc], [yc], color="yellow", marker="*", s=150, edgecolor="black", zorder=4, label="Centroid")
        
        ax.set_title("Delaunay Triangulation Mesh: Vacancy Regions form Large Empty Triangles", fontsize=13)
        ax.legend()
        ax.grid(False)
        ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
    except Exception as e:
        print(f"Delaunay visualization failed: {e}")
else:
    print("ERROR: Sublattice not found.")


In [ ]:
# Visualizing Corrected Coordinate-Independent Strain Vectors (Edge-Filtered)
print("Calculating corrected lattice strain with Convex Hull boundary filtering...")

if "sublattice" in locals():
    try:
        from scipy.spatial import Delaunay, ConvexHull
        from scipy.interpolate import griddata
        import matplotlib.pyplot as plt
        
        # 1. Get refined atomic coordinates
        points = np.array([p.get_pixel_position() for p in sublattice.atom_list])
        if len(points) < 10: raise ValueError("Insufficient atoms for analysis.")
        
        xc = np.mean(points[:, 0])
        yc = np.mean(points[:, 1])
        
        # 2. Identify boundary atoms using Convex Hull
        hull = ConvexHull(points)
        boundary_mask = np.zeros(len(points), dtype=bool)
        boundary_mask[hull.vertices] = True
        
        # 3. Compute Delaunay Triangulation
        tri = Delaunay(points)
        
        # 4. Calculate average reference lattice spacing (d0) from all edges
        edge_lengths = []
        for simplex in tri.simplices:
            nodes = points[simplex]
            edge_lengths.append(np.linalg.norm(nodes[0] - nodes[1]))
            edge_lengths.append(np.linalg.norm(nodes[1] - nodes[2]))
            edge_lengths.append(np.linalg.norm(nodes[2] - nodes[0]))
        d0 = np.mean(edge_lengths)
        print(f"Reference lattice spacing (d0): {d0:.2f} px")
        
        # Expected coordinate standard deviation for an equilateral triangle of side d0
        ref_std = d0 / np.sqrt(6.0)
        
        # 5. Calculate local strain components on triangles, filtering out surface triangles
        exx_list = []
        eyy_list = []
        valid_simplices = []
        
        for simplex in tri.simplices:
            # Skip surface triangles that connect 2 or more boundary atoms
            num_boundary = sum(boundary_mask[idx] for idx in simplex)
            if num_boundary >= 2:
                continue
                
            nodes = points[simplex]
            exx_list.append((np.std(nodes[:, 0]) - ref_std) / ref_std)
            eyy_list.append((np.std(nodes[:, 1]) - ref_std) / ref_std)
            valid_simplices.append(simplex)
            
        exx_list = np.array(exx_list)
        eyy_list = np.array(eyy_list)
        valid_simplices = np.array(valid_simplices)
        
        if len(valid_simplices) == 0:
            raise ValueError("No interior triangles remaining after boundary filtering.")
            
        # 6. Interpolate strain values back onto individual atom points
        tri_centers = points[valid_simplices].mean(axis=1)
        exx_atoms = griddata(tri_centers, exx_list, points, method="linear", fill_value=0)
        eyy_atoms = griddata(tri_centers, eyy_list, points, method="linear", fill_value=0)
        
        # 7. Project strain onto the Radial unit vector at each atom site
        err_list = []
        u_arrow = []
        v_arrow = []
        
        for i in range(len(points)):
            x, y = points[i]
            dx = x - xc
            dy = y - yc
            r = np.sqrt(dx**2 + dy**2 + 1e-12)
            
            # Unit radial vector
            ux = dx / r
            uy = dy / r
            
            # Radial strain projection
            err = exx_atoms[i] * (ux**2) + eyy_atoms[i] * (uy**2)
            
            # Set strain and arrows to zero for boundary surface atoms to remove artifacts
            if boundary_mask[i]:
                err = 0.0
                ux = 0.0
                uy = 0.0
                
            err_list.append(err)
            u_arrow.append(err * ux)
            v_arrow.append(err * uy)
            
        err_list = np.array(err_list)
        u_arrow = np.array(u_arrow)
        v_arrow = np.array(v_arrow)
        
        # 8. Visualization: Single Corrected Quiver Plot without edge artifacts
        fig, ax = plt.subplots(figsize=(10, 8))
        
        # Plot the raw image
        ax.imshow(s.data, cmap="gray", origin="upper")
        ax.scatter(points[:, 0], points[:, 1], color="gray", s=5, alpha=0.5, label="Atoms")
        ax.scatter([xc], [yc], color="yellow", marker="*", s=150, edgecolor="black", label="Centroid")
        
        # Plot quiver vectors
        q = ax.quiver(points[:, 0], points[:, 1], u_arrow, v_arrow, err_list * 100,
                      cmap="turbo", angles="xy", scale_units="xy", scale=0.01,
                      width=0.008, headwidth=4, headlength=5, headaxislength=4.5,
                      clim=(-10, 10))
        
        plt.colorbar(q, ax=ax, fraction=0.038, pad=0.04, label="Radial Strain [%]")
        ax.legend()
        ax.grid(False)
        ax.axis('off')
        
        # Add scale bar (2 nm)
        px_val = globals().get('px_size_A', 0.215)
        ny, nx = s.data.shape
        bar_w = 20.0 / px_val
        bar_h = max(1.0, ny * 0.02)
        
        x_pos = nx - (nx * 0.08) - bar_w
        y_pos = ny - (ny * 0.08) - bar_h
        
        from matplotlib.patches import Rectangle
        rect = Rectangle((x_pos, y_pos), bar_w, bar_h, color='white', fill=True, zorder=5)
        ax.add_patch(rect)
        
        ax.text(x_pos + bar_w/2.0, y_pos - (ny * 0.015), "2 nm", 
                color='white', fontsize=12, fontweight='bold', 
                ha='center', va='bottom', zorder=5)
        
        ax.set_title("Corrected Radial Strain Vectors (Convex Hull Edge-Filtered)", fontsize=13)
        plt.tight_layout()
        plt.show()
        
    except Exception as e:
        print(f"Corrected strain analysis failed: {e}")
else:
    print("ERROR: Sublattice not found.")


In [ ]:
# Step 5: Voronoi Cell Analysis (Local Area / Volumetric Strain)
print("Calculating local lattice density using Voronoi cells...")

if "sublattice" in locals():
    try:
        from scipy.spatial import Voronoi, voronoi_plot_2d
        from scipy.interpolate import griddata
        
        # 1. Get refined atomic coordinates
        points = np.array([p.get_pixel_position() for p in sublattice.atom_list])
        if len(points) < 10: raise ValueError("Insufficient atoms for Voronoi analysis.")
        
        # 2. Compute Voronoi Tesselation
        # Each Voronoi cell represents the territory of a single atom
        vor = Voronoi(points)
        
        # 3. Calculate the area of each finite Voronoi cell
        areas = []
        valid_points = []
        
        for i, region_idx in enumerate(vor.point_region):
            region = vor.regions[region_idx]
            # Skip infinite regions (atoms on the edge of the grain)
            if -1 not in region and len(region) > 0:
                # Shoelace formula for polygon area
                verts = vor.vertices[region]
                area = 0.5 * np.abs(np.dot(verts[:, 0], np.roll(verts[:, 1], 1)) - 
                                    np.dot(verts[:, 1], np.roll(verts[:, 0], 1)))
                areas.append(area)
                valid_points.append(points[i])
        
        areas = np.array(areas)
        valid_points = np.array(valid_points)
        
        # 4. Calculate Volumetric Strain (Area deviation from mean)
        avg_area = np.mean(areas)
        area_strain = (areas - avg_area) / avg_area * 100 # % relative expansion/contraction
        
        # 5. Interpolate for continuous visualization
        grid_y, grid_x = np.mgrid[0:s.data.shape[0], 0:s.data.shape[1]]
        strain_map = griddata(valid_points, area_strain, (grid_x, grid_y), method="linear")
        
        # 6. Visualization
        fig, axes = plt.subplots(1, 2, figsize=(16, 7))
        
        # Left: Voronoi Diagram Overlay
        ax_vor = axes[0]
        ax_vor.imshow(s.data, cmap="gray", origin="upper")
        ax_vor.scatter(points[:, 0], points[:, 1], color="red", s=5, alpha=0.5, label="Atoms")
        voronoi_plot_2d(vor, ax=ax_vor, show_vertices=False, line_colors="yellow", line_width=1, point_size=0)
        
        ax_vor.set_title("Voronoi Cells (Atomic Territory)")
        ax_vor.set_xlim(0, s.data.shape[1])
        ax_vor.set_ylim(s.data.shape[0], 0)
        
        # Right: Volumetric (Area) Strain Map
        im = axes[1].imshow(strain_map, cmap="RdBu_r", vmin=-5, vmax=5, origin="upper")
        axes[1].set_title("Volumetric Strain (Area Change) [%]")
        plt.colorbar(im, ax=axes[1], label="Area expansion (+) / contraction (-) [%]")
        
        plt.suptitle("Voronoi-Based Lattice Density Analysis", fontsize=16)
        plt.tight_layout()
        plt.show()
        
    except Exception as e:
        print(f"Voronoi analysis failed: {e}")
else:
    print("ERROR: Sublattice not found.")
